In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display

import plotly.io as pio
pio.templates.default = "plotly_white"

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
df = pd.read_csv("netflix_titles.csv")
display(df.head())
print("Dataset Shape:", df.shape)
df.info()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


Dataset Shape: (8807, 12)
<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [3]:
df = df.copy()

df["date_added"] = pd.to_datetime(
    df["date_added"].str.strip(),
    format="mixed",
    errors="coerce"
)

df["duration_minutes"] = pd.to_numeric(
    df.loc[df["type"] == "Movie", "duration"]
      .str.replace(" min", "", regex=False),
    errors="coerce"
)

df["seasons"] = pd.to_numeric(
    df.loc[df["type"] == "TV Show", "duration"]
      .str.replace(" Seasons", "", regex=False)
      .str.replace(" Season", "", regex=False),
    errors="coerce"
)

df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month_name()

print("Data preprocessing completed!")
display(df.head())

Data preprocessing completed!


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_minutes,seasons,year_added,month_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",90.0,NaN,2021.0,September
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",NaN,2.0,2021.0,September
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,NaN,1.0,2021.0,September
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",NaN,1.0,2021.0,September
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,NaN,2.0,2021.0,September


In [4]:
content_counts = df["type"].value_counts().reset_index()

content_counts.columns = ["Content Type", "Count"]

fig1 = px.pie(
    content_counts,
    names="Content Type",
    values="Count",
    title="Netflix Content Distribution",
    hole=0.45,
    color_discrete_sequence=["#E50914", "#221F1F"]
)

fig1.update_traces(
    textinfo="percent+label",
    hovertemplate="<b>%{label}</b><br>Titles: %{value}<br>Share: %{percent}<extra></extra>"
)

fig1.update_layout(
    template="plotly_white",
    title_x=0.5,
    font=dict(family="Arial", size=13),
    height=450
)

fig1.show()

In [5]:
yearly_content = (
    df.dropna(subset=["year_added"])
      .groupby("year_added")
      .size()
      .reset_index(name="Total Titles")
)

fig2 = px.line(
    yearly_content,
    x="year_added",
    y="Total Titles",
    title="Netflix Content Added Over the Years",
    markers=True,
    color_discrete_sequence=["#E50914"],
    labels={
        "year_added": "Year",
        "Total Titles": "Number of Titles Added"
    }
)

fig2.update_traces(
    line=dict(width=3),
    hovertemplate="Year: %{x}<br>Titles Added: %{y}<extra></extra>"
)

fig2.update_layout(
    template="plotly_white",
    title_x=0.5,
    font=dict(family="Arial", size=13),
    height=450,
    hovermode="x unified"
)

fig2.show()

In [6]:
movies = df[df["type"] == "Movie"].copy()

movies = movies.dropna(subset=["duration_minutes"])

fig3 = px.histogram(
    movies,
    x="duration_minutes",
    nbins=30,
    title="Distribution of Netflix Movie Durations",
    color_discrete_sequence=["#E50914"],
    labels={
        "duration_minutes": "Movie Duration (Minutes)",
        "count": "Number of Movies"
    }
)

fig3.update_traces(
    hovertemplate="Duration: %{x}<br>Number of Movies: %{y}<extra></extra>"
)

fig3.update_layout(
    template="plotly_white",
    title_x=0.5,
    font=dict(family="Arial", size=13),
    height=450,
    bargap=0.1
)

fig3.show()

In [7]:
rating_movies = movies.dropna(
    subset=["duration_minutes", "rating"]
)

top_ratings = rating_movies["rating"].value_counts().head(8).index

rating_movies = rating_movies[
    rating_movies["rating"].isin(top_ratings)
]

fig4 = px.box(
    rating_movies,
    x="rating",
    y="duration_minutes",
    color="rating",
    title="Movie Duration Distribution by Content Rating",
    labels={
        "rating": "Content Rating",
        "duration_minutes": "Duration (Minutes)"
    },
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig4.update_layout(
    template="plotly_white",
    title_x=0.5,
    font=dict(family="Arial", size=13),
    height=500,
    showlegend=False
)

fig4.update_traces(
    hovertemplate="Rating: %{x}<br>Duration: %{y} minutes<extra></extra>"
)

fig4.show()

In [8]:
scatter_data = movies.dropna(
    subset=["release_year", "duration_minutes"]
)

fig5 = px.scatter(
    scatter_data,
    x="release_year",
    y="duration_minutes",
    color="rating",
    title="Movie Release Year vs Duration",
    labels={
        "release_year": "Release Year",
        "duration_minutes": "Duration (Minutes)",
        "rating": "Content Rating"
    },
    hover_data=["title"],
    opacity=0.6
)

fig5.update_layout(
    template="plotly_white",
    title_x=0.5,
    font=dict(family="Arial", size=13),
    height=500
)

fig5.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "Release Year: %{x}<br>"
        "Duration: %{y} minutes<br>"
        "Rating: %{legendgroup}<extra></extra>"
    )
)

fig5.show()

In [9]:
heatmap_data = df.dropna(
    subset=["year_added", "month_added"]
).copy()

monthly_content = (
    heatmap_data.groupby(
        ["year_added", "month_added"]
    )
    .size()
    .reset_index(name="Total Titles")
)

month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

monthly_content["month_added"] = pd.Categorical(
    monthly_content["month_added"],
    categories=month_order,
    ordered=True
)

heatmap_pivot = monthly_content.pivot(
    index="month_added",
    columns="year_added",
    values="Total Titles"
).fillna(0)

fig6 = px.imshow(
    heatmap_pivot,
    labels=dict(
        x="Year",
        y="Month",
        color="Titles Added"
    ),
    title="Netflix Content Addition by Month and Year",
    color_continuous_scale="Reds",
    aspect="auto"
)

fig6.update_layout(
    template="plotly_white",
    title_x=0.5,
    font=dict(family="Arial", size=13),
    height=500
)

fig6.show()

In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output

content_filter = widgets.Dropdown(
    options=["All Content", "Movie", "TV Show"],
    value="All Content",
    description="Filter:",
    style={"description_width": "initial"}
)

dashboard_output = widgets.Output()

display(content_filter)
display(dashboard_output)

Dropdown(description='Filter:', options=('All Content', 'Movie', 'TV Show'), style=DescriptionStyle(descriptio…

Output()

In [11]:
def update_dashboard(change=None):

    selected_type = content_filter.value

    if selected_type == "All Content":
        filtered_df = df.copy()
    else:
        filtered_df = df[df["type"] == selected_type].copy()

    with dashboard_output:
        clear_output(wait=True)

        yearly = (
            filtered_df.dropna(subset=["year_added"])
            .groupby("year_added")
            .size()
            .reset_index(name="Total Titles")
        )

        fig_year = px.line(
            yearly,
            x="year_added",
            y="Total Titles",
            markers=True,
            title=f"Content Added Over the Years - {selected_type}",
            color_discrete_sequence=["#E50914"]
        )

        fig_year.update_layout(
            template="plotly_white",
            title_x=0.5,
            height=400
        )

        display(fig_year)

        monthly = (
            filtered_df.dropna(
                subset=["year_added", "month_added"]
            )
            .groupby(["month_added", "year_added"])
            .size()
            .reset_index(name="Total Titles")
        )

        month_order = [
            "January", "February", "March", "April",
            "May", "June", "July", "August",
            "September", "October", "November", "December"
        ]

        monthly["month_added"] = pd.Categorical(
            monthly["month_added"],
            categories=month_order,
            ordered=True
        )

        pivot = monthly.pivot(
            index="month_added",
            columns="year_added",
            values="Total Titles"
        ).fillna(0)

        fig_heat = px.imshow(
            pivot,
            title=f"Monthly Content Additions - {selected_type}",
            color_continuous_scale="Reds",
            labels={
                "x": "Year",
                "y": "Month",
                "color": "Titles Added"
            },
            aspect="auto"
        )

        fig_heat.update_layout(
            template="plotly_white",
            title_x=0.5,
            height=450
        )

        display(fig_heat)


content_filter.observe(update_dashboard, names="value")

update_dashboard()

In [12]:
total_titles = len(df)
total_movies = (df["type"] == "Movie").sum()
total_tv_shows = (df["type"] == "TV Show").sum()
total_countries = df["country"].dropna().str.split(", ").explode().nunique()

from IPython.display import HTML, display

display(HTML("""
<div style="
    background-color: #141414;
    padding: 25px;
    border-radius: 12px;
    text-align: center;
    color: white;
    margin-bottom: 20px;
">
    <h1 style="color: #E50914; margin: 0;">
        NETFLIX CONTENT ANALYTICS
    </h1>
    <p style="margin: 8px 0 0;">
        Interactive Dashboard | Python & Plotly
    </p>
</div>
"""))

kpi_html = f"""
<div style="display: flex; gap: 12px; flex-wrap: wrap;">

    <div style="background:#222; color:white; padding:20px;
    border-radius:10px; flex:1; min-width:140px; text-align:center;">
        <h3>Total Titles</h3>
        <h2 style="color:#E50914;">{total_titles:,}</h2>
    </div>

    <div style="background:#222; color:white; padding:20px;
    border-radius:10px; flex:1; min-width:140px; text-align:center;">
        <h3>Movies</h3>
        <h2 style="color:#E50914;">{total_movies:,}</h2>
    </div>

    <div style="background:#222; color:white; padding:20px;
    border-radius:10px; flex:1; min-width:140px; text-align:center;">
        <h3>TV Shows</h3>
        <h2 style="color:#E50914;">{total_tv_shows:,}</h2>
    </div>

    <div style="background:#222; color:white; padding:20px;
    border-radius:10px; flex:1; min-width:140px; text-align:center;">
        <h3>Countries</h3>
        <h2 style="color:#E50914;">{total_countries:,}</h2>
    </div>

</div>
"""

display(HTML(kpi_html))

In [14]:
from plotly.subplots import make_subplots

In [15]:
dashboard = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{"type": "domain"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}]
    ],
    subplot_titles=[
        "Movies vs TV Shows",
        "Content Added Over the Years",
        "Movie Duration Distribution",
        "Movie Duration by Rating",
        "Release Year vs Movie Duration",
        "Monthly Content Addition Heatmap"
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.12
)

dashboard.add_trace(
    go.Pie(
        labels=content_counts["Content Type"],
        values=content_counts["Count"],
        hole=0.45,
        marker=dict(colors=["#E50914", "#221F1F"]),
        name="Content Type"
    ),
    row=1, col=1
)

dashboard.add_trace(
    go.Scatter(
        x=yearly_content["year_added"],
        y=yearly_content["Total Titles"],
        mode="lines+markers",
        name="Titles Added",
        line=dict(color="#E50914", width=3)
    ),
    row=1, col=2
)

dashboard.add_trace(
    go.Histogram(
        x=movies["duration_minutes"],
        nbinsx=30,
        name="Movie Duration",
        marker_color="#E50914"
    ),
    row=2, col=1
)

dashboard.add_trace(
    go.Box(
        x=rating_movies["rating"],
        y=rating_movies["duration_minutes"],
        name="Rating",
        marker_color="#E50914",
        showlegend=False
    ),
    row=2, col=2
)

dashboard.add_trace(
    go.Scatter(
        x=scatter_data["release_year"],
        y=scatter_data["duration_minutes"],
        mode="markers",
        name="Movies",
        marker=dict(
            color="#E50914",
            size=6,
            opacity=0.6
        ),
        text=scatter_data["title"],
        hovertemplate=(
            "Title: %{text}<br>"
            "Release Year: %{x}<br>"
            "Duration: %{y} minutes"
            "<extra></extra>"
        )
    ),
    row=3, col=1
)

dashboard.add_trace(
    go.Heatmap(
        z=heatmap_pivot.values,
        x=heatmap_pivot.columns.astype(str),
        y=heatmap_pivot.index,
        colorscale="Reds",
        name="Titles Added",
        colorbar=dict(title="Titles")
    ),
    row=3, col=2
)

dashboard.update_layout(
    title=dict(
        text="Netflix Content Analytics Dashboard",
        x=0.5,
        font=dict(size=24, color="#E50914")
    ),
    template="plotly_white",
    height=1500,
    width=1100,
    showlegend=False,
    font=dict(family="Arial", size=11),
    margin=dict(t=100, b=50, l=60, r=60)
)

dashboard.update_xaxes(title_text="Year", row=1, col=2)
dashboard.update_yaxes(title_text="Number of Titles", row=1, col=2)

dashboard.update_xaxes(title_text="Duration (Minutes)", row=2, col=1)
dashboard.update_yaxes(title_text="Number of Movies", row=2, col=1)

dashboard.update_xaxes(title_text="Content Rating", row=2, col=2)
dashboard.update_yaxes(title_text="Duration (Minutes)", row=2, col=2)

dashboard.update_xaxes(title_text="Release Year", row=3, col=1)
dashboard.update_yaxes(title_text="Duration (Minutes)", row=3, col=1)

dashboard.update_layout(
    hoverlabel=dict(
        bgcolor="white",
        font_size=12
    )
)

dashboard.show()

In [16]:
from plotly.subplots import make_subplots

final_dashboard = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{"type": "domain"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}]
    ],
    subplot_titles=[
        "Content Distribution",
        "Content Added Over the Years",
        "Movie Duration Distribution",
        "Movie Duration by Rating",
        "Release Year vs Duration",
        "Monthly Content Additions"
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.12
)

month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

filter_options = ["All Content", "Movie", "TV Show"]
visibility_groups = []

for option in filter_options:

    if option == "All Content":
        data = df.copy()
    else:
        data = df[df["type"] == option].copy()

    counts = data["type"].value_counts()

    final_dashboard.add_trace(
        go.Pie(
            labels=counts.index,
            values=counts.values,
            hole=0.45,
            marker=dict(colors=["#E50914", "#221F1F"]),
            name="Content Type",
            hovertemplate="%{label}: %{value}<br>%{percent}<extra></extra>"
        ),
        row=1, col=1
    )

    yearly = (
        data.dropna(subset=["year_added"])
        .groupby("year_added")
        .size()
        .reset_index(name="Total Titles")
    )

    final_dashboard.add_trace(
        go.Scatter(
            x=yearly["year_added"],
            y=yearly["Total Titles"],
            mode="lines+markers",
            line=dict(color="#E50914", width=3),
            name="Titles Added",
            hovertemplate="Year: %{x}<br>Titles: %{y}<extra></extra>"
        ),
        row=1, col=2
    )

    movie_data = data[
        data["type"] == "Movie"
    ].dropna(subset=["duration_minutes"])

    final_dashboard.add_trace(
        go.Histogram(
            x=movie_data["duration_minutes"],
            nbinsx=30,
            marker_color="#E50914",
            name="Movie Duration"
        ),
        row=2, col=1
    )

    box_data = movie_data.dropna(subset=["rating"])

    final_dashboard.add_trace(
        go.Box(
            x=box_data["rating"],
            y=box_data["duration_minutes"],
            marker_color="#E50914",
            name="Duration by Rating"
        ),
        row=2, col=2
    )

    scatter = movie_data.dropna(
        subset=["release_year"]
    )

    final_dashboard.add_trace(
        go.Scatter(
            x=scatter["release_year"],
            y=scatter["duration_minutes"],
            mode="markers",
            marker=dict(color="#E50914", size=6, opacity=0.6),
            text=scatter["title"],
            name="Movies",
            hovertemplate=(
                "Title: %{text}<br>"
                "Year: %{x}<br>"
                "Duration: %{y} minutes<extra></extra>"
            )
        ),
        row=3, col=1
    )

    monthly = (
        data.dropna(subset=["year_added", "month_added"])
        .groupby(["month_added", "year_added"])
        .size()
        .reset_index(name="Total Titles")
    )

    monthly["month_added"] = pd.Categorical(
        monthly["month_added"],
        categories=month_order,
        ordered=True
    )

    pivot = monthly.pivot(
        index="month_added",
        columns="year_added",
        values="Total Titles"
    ).reindex(month_order).fillna(0)

    final_dashboard.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=pivot.columns.astype(str),
            y=pivot.index,
            colorscale="Reds",
            colorbar=dict(title="Titles"),
            name="Monthly Additions"
        ),
        row=3, col=2
    )

    group_visibility = [False] * 18
    group_index = len(visibility_groups)

    for i in range(6):
        group_visibility[group_index * 6 + i] = True

    visibility_groups.append(group_visibility)


buttons = []

for option, visibility in zip(
    filter_options, visibility_groups
):
    buttons.append(
        dict(
            label=option,
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"Netflix Content Analytics - {option}"}
            ]
        )
    )


final_dashboard.update_layout(
    title=dict(
        text="Netflix Content Analytics Dashboard - All Content",
        x=0.5,
        font=dict(size=24, color="#E50914")
    ),
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.01,
            y=1.12,
            xanchor="left",
            yanchor="top"
        )
    ],
    template="plotly_white",
    height=1550,
    width=1150,
    showlegend=False,
    font=dict(family="Arial", size=12),
    margin=dict(t=160, b=60, l=60, r=60)
)

final_dashboard.update_xaxes(title_text="Year", row=1, col=2)
final_dashboard.update_yaxes(title_text="Titles Added", row=1, col=2)

final_dashboard.update_xaxes(title_text="Duration (Minutes)", row=2, col=1)
final_dashboard.update_yaxes(title_text="Number of Movies", row=2, col=1)

final_dashboard.update_xaxes(title_text="Rating", row=2, col=2)
final_dashboard.update_yaxes(title_text="Duration (Minutes)", row=2, col=2)

final_dashboard.update_xaxes(title_text="Release Year", row=3, col=1)
final_dashboard.update_yaxes(title_text="Duration (Minutes)", row=3, col=1)

final_dashboard.show()

In [17]:
final_dashboard.write_html(
    "dashboard.html",
    include_plotlyjs=True,
    full_html=True
)

print("Dashboard exported successfully!")

Dashboard exported successfully!
